# Análise de Acessos de Telefonia Móvel no Brasil (Anatel)

Este notebook apresenta uma análise exploratória dos acessos de telefonia móvel no Brasil. A maior parte das transformações foi movida para o módulo `src/telefonia/analysis.py`, deixando o notebook focado na leitura dos resultados e nas visualizações.


## 1. Configuração do ambiente

As bibliotecas e funções do projeto são carregadas abaixo. O caminho do repositório é resolvido de forma relativa, evitando caminhos absolutos de uma máquina específica.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from telefonia.analysis import (  # noqa: E402
    DATA_FILE,
    load_telefonia_data,
    summarize_by_state,
    summarize_by_technology,
    summarize_time_evolution,
)

sns.set_theme(style="whitegrid", context="notebook")


## 2. Carregamento e inspeção dos dados

O arquivo principal fica em `data/raw/br_anatel_telefonia_movel_ddd.csv`. A função `load_telefonia_data` carrega o CSV e cria a coluna mensal `data`.


In [ ]:
df_tel = load_telefonia_data(DATA_FILE)
df_tel.head()


In [ ]:
df_tel.info()


In [ ]:
df_tel.describe(include="all")


## 3. Evolução temporal geral

A série abaixo soma os acessos por mês e converte o resultado para milhões, facilitando a leitura do eixo Y.


In [ ]:
df_evolucao = summarize_time_evolution(df_tel)
df_evolucao.head()


In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df_evolucao,
    x="data",
    y="acessos_milhoes",
    marker="o",
    color="b",
    linewidth=2,
)

plt.title("Evolução temporal do número total de acessos", fontsize=16)
plt.xlabel("Ano", fontsize=12)
plt.ylabel("Total de acessos (em milhões)", fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


## 4. Transição tecnológica

A próxima visualização compara a evolução das principais tecnologias de rede presentes no conjunto de dados.


In [ ]:
df_tec_filtrado = summarize_by_technology(df_tel)
df_tec_filtrado.head()


In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df_tec_filtrado,
    x="data",
    y="acessos_milhoes",
    hue="tecnologia",
    marker="o",
    linewidth=2,
)

plt.title("Transição tecnológica: evolução dos acessos no Brasil", fontsize=16)
plt.xlabel("Ano", fontsize=12)
plt.ylabel("Total de acessos (em milhões)", fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.7)
plt.legend(title="Tecnologia", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 5. Distribuição geográfica por UF

A análise abaixo agrega os acessos por unidade federativa para um ano específico.


In [ ]:
ANALYSIS_YEAR = 2020

df_uf = summarize_by_state(df_tel, ANALYSIS_YEAR)
df_uf.head()


In [ ]:
plt.figure(figsize=(12, 10))
sns.barplot(data=df_uf, x="acessos_milhoes", y="sigla_uf")

plt.title(f"Distribuição total de acessos por estado (UF) em {ANALYSIS_YEAR}", fontsize=16)
plt.xlabel("Total de acessos (em milhões)", fontsize=12)
plt.ylabel("Estado (UF)", fontsize=12)
plt.grid(True, axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


## 6. Próximos passos sugeridos

- Salvar as figuras em `reports/figures` quando forem usadas em apresentações ou relatórios.
- Criar novas funções em `src/telefonia/analysis.py` para análises adicionais.
- Adicionar testes automatizados para validar as agregações.
